In [ ]:
%matplotlib inline
import os, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
warnings.filterwarnings('ignore')

plt.rcParams['font.family']        = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

OFF_BLACK      = '#1E1E1E'
COOL_GREY      = '#C8CDD6'
SATURATED_BLUE = '#245BFF'
LIGHT_GREY     = '#EEF1F5'
WHITE          = '#FFFFFF'
ACCENT_RED     = '#E63946'

ROOT    = '/workspace/ST-GNN Modeling'
OUT     = os.path.join(ROOT, 'Visualization/outputs')
FIG_DPI = 200
os.makedirs(OUT, exist_ok=True)

print('설정 완료')

In [ ]:
# ── 데이터 ───────────────────────────────────────────────────────────────

stgnn_data = [
    ('Climatological (baseline)', 3.2026, False),
    ('S1 Static (PM10 only)',     2.8123, False),
    ('S6 Static (PM10+Pollutants+Mask)', 2.7672, False),
    ('S7 Static (PM10+Weather)', 2.6985, False),
    ('S3 Static (PM10+Pollutants)', 2.6144, True),
]

hs = pd.read_csv(os.path.join(ROOT, 'HiddenExtension_V5/checkpoints/holdout_summary.csv'))
he_nm = {
    'V5-base':         'V5-base (MLP only)',
    'V5-monthly':      'V5-monthly (+Monthly FE)',
    'V5-monthly-hour': 'V5-monthly-hour (+Monthly+Hour)',
    'V5-bias':         'V5-bias (+Station Bias)',
    'V5-season':       'V5-season (+Season FE)',
    'V5-hier':         'V5-hier (+Hierarchical)',
}
he_data = [
    (he_nm[r['exp_id']], r['holdout_MAE'], r['exp_id'] == 'V5-base')
    for _, r in hs.sort_values('holdout_MAE').iterrows()
]

with open(os.path.join(ROOT, 'RoadExtension_V3/checkpoints/ablation_results.json')) as f:
    v3 = json.load(f)['results']
rd_nm = {
    'TwoStage':         'TwoStage (Temporal+Spatial)',
    'LightGBM_BC':      'LightGBM BC (Box-Cox)',
    'XGBoost_BC':       'XGBoost BC (Box-Cox)',
    'LightGBM_Weighted':'LightGBM (Sample Weight)',
    'LightGBM_Huber':   'LightGBM Huber (Huber Loss)',
}
rd_data = [(rd_nm[k], v3[k]['test_mae'], k == 'TwoStage')
           for k in rd_nm if k in v3]
rd_data.sort(key=lambda x: x[1])

print('데이터 로드 완료')

In [ ]:
# ── Connected Dot Plot + 시각화 ───────────────────────────────────────────

def draw_dotplot(ax, data, xlabel, title, subtitle):
    labels = [d[0] for d in data]
    vals   = [d[1] for d in data]
    chosen = [d[2] for d in data]
    best_v = vals[chosen.index(True)]
    y      = np.arange(len(data))

    ax.set_facecolor(WHITE)
    for sp in ax.spines.values(): sp.set_visible(False)
    ax.spines['bottom'].set_visible(True)
    ax.spines['bottom'].set_edgecolor(COOL_GREY)
    ax.spines['bottom'].set_linewidth(0.7)
    ax.tick_params(colors=OFF_BLACK, labelsize=8.5, length=0)
    ax.tick_params(axis='y', pad=6)
    ax.set_axisbelow(True)
    ax.grid(axis='x', color=COOL_GREY, linewidth=0.4, alpha=0.5, zorder=0)

    ax.axvline(best_v, color=SATURATED_BLUE, linewidth=1.5,
               linestyle='--', alpha=0.7, zorder=2)

    for i, (v, c) in enumerate(zip(vals, chosen)):
        bad   = (v - best_v) / best_v > 0.10
        lc    = ACCENT_RED if bad else COOL_GREY
        ax.plot([best_v, v], [i, i], color=lc,
                linewidth=1.4, alpha=0.6, zorder=3)
        if c:
            ax.scatter(v, i, s=130, color=SATURATED_BLUE,
                       edgecolors=WHITE, linewidths=1.5, zorder=5)
        else:
            ax.scatter(v, i, s=70, color=lc,
                       edgecolors=WHITE, linewidths=1.2, zorder=5)

    x_range = max(vals) - min(vals)
    for i, (v, c) in enumerate(zip(vals, chosen)):
        if c:
            ax.text(v - x_range*0.02, i,
                    '{:.4f}  BEST'.format(v),
                    va='center', ha='right', fontsize=8.5,
                    fontweight='bold', color=SATURATED_BLUE)
        else:
            delta = (v - best_v) / best_v * 100
            sign  = '+' if delta >= 0 else ''
            color = ACCENT_RED if delta > 10 else OFF_BLACK
            ax.text(v + x_range*0.02, i,
                    '{}{:.1f}%'.format(sign, delta),
                    va='center', ha='left', fontsize=8, color=color)

    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8.5)
    ax.invert_yaxis()
    pad = x_range * 0.38
    ax.set_xlim(min(vals) - pad, max(vals) + pad)
    ax.set_xlabel(xlabel, fontsize=9, color=OFF_BLACK, labelpad=4)
    ax.set_title(title, fontsize=11, fontweight='bold',
                 color=OFF_BLACK, pad=14)
    ax.text(0.5, 1.055, subtitle, transform=ax.transAxes,
            ha='center', va='bottom', fontsize=8, color=COOL_GREY)


fig = plt.figure(figsize=(16, 5.8), dpi=FIG_DPI)
fig.patch.set_facecolor(WHITE)
gs = gridspec.GridSpec(1, 3, figure=fig,
                       left=0.14, right=0.98,
                       top=0.80, bottom=0.13,
                       wspace=0.60)
ax1, ax2, ax3 = [fig.add_subplot(gs[i]) for i in range(3)]

draw_dotplot(ax1, stgnn_data,
             'Test MAE (ug/m3)', '① ST-GNN', 'window=12  ·  test MAE')
draw_dotplot(ax2, he_data,
             'Holdout MAE (ug/m3)', '② HiddenExtension V5',
             'Geographic cluster holdout MAE')
draw_dotplot(ax3, rd_data,
             'Test MAE (ug/m3)', '③ RoadExtension V3', '2025 test MAE')

fig.text(0.56, 0.96,
         'ST-GNN Pipeline  -  Model Performance Summary',
         ha='center', va='top', fontsize=13,
         fontweight='bold', color=OFF_BLACK)

handles = [
    mpatches.Patch(color=SATURATED_BLUE, label='Final Model'),
    mpatches.Patch(color=COOL_GREY,      label='Comparison'),
    mpatches.Patch(color=ACCENT_RED,     label='Comparison (>10% worse)'),
]
fig.legend(handles=handles, loc='lower center', ncol=3,
           fontsize=8.5, facecolor=WHITE, edgecolor=COOL_GREY,
           bbox_to_anchor=(0.56, -0.02))

plt.savefig(os.path.join(OUT, 'fig_performance_summary.png'),
            dpi=FIG_DPI, bbox_inches='tight',
            facecolor=WHITE, pad_inches=0.10)
print('Saved:', os.path.join(OUT, 'fig_performance_summary.png'))
plt.show()